# Table 5 augmentation - ONE notebook for Kaggle (GPU T4, Internet ON)
Runs: (1) shared helpers, (2) RegulationQA retrieval cache (~30 min, skipped if shards exist in /kaggle/working),
(3) controlled single vs adaptive study + BERTScore, (4) Qwen/Mistral judges on the adaptive spans (set RUN_JUDGES below).
Add as Input datasets: the RegulationQA repo folders (index/, dataset/, evaluation/). Nothing else to upload.
If the session stops, re-run all cells: finished cache shards are skipped.

In [1]:
RUN_JUDGES = True      # False = stop after step 12 (no Qwen/Mistral)

## Shared helpers (adsir_common)

In [2]:
"""
adsir_common.py  --  shared configuration and helper functions for the ADSIR / RegulationQA code series.

EVERY numbered step (01_...py to 11_...py) starts with:   from adsir_common import *
so that a setting (paths, pool size, thresholds, model names) is defined in exactly ONE place.

Sections
  1. Paths and file discovery (works on Kaggle and on a local checkout of the repo)
  2. Pipeline constants (the paper's configuration)
  3. Text helpers: tokenizer, token F1, inner decay, RRF fusion
  4. RegulationQA label normalisation (single-source vs cross-document, pooled variants)
  5. Statistics helper: paired bootstrap
"""
import os, re, sys, json, glob, pickle, string, time, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------------------------
IS_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path("./outputs")      # everything we write goes here
WORK.mkdir(parents=True, exist_ok=True)

# Extra places to look for artifacts when running locally (the repo layout).
REPO_DIRS = [Path(p) for p in ["index", "dataset", "evaluation", "../index", "../dataset", "../evaluation", "."]]

def find(name, required=True):
    """Locate a file by name: first in WORK, then anywhere under /kaggle/input, then in the repo folders."""
    cands = [WORK / name]
    if IS_KAGGLE: cands += list(Path("/kaggle/input").rglob(name))
    cands += [d / name for d in REPO_DIRS]
    for c in cands:
        if Path(c).exists():
            print(f"[find] {name:32s} -> {c}")
            return str(c)
    if required: raise FileNotFoundError(f"{name} not found. Add the dataset that contains it to this notebook, or run the step that creates it.")
    return None

# ---------------------------------------------------------------------------------------------
# 2. PIPELINE CONSTANTS  (the configuration used in the paper)
# ---------------------------------------------------------------------------------------------
RRF_K          = 60       # Reciprocal Rank Fusion constant, rank is 0-based so the term is 1/(RRF_K + rank + 1)
PRE_K          = 100      # passages taken from EACH retriever (BGE and BM25) before fusion
M_RQA          = 20       # fused candidates scored by the cross-encoder on RegulationQA (paper text: m = 20)
M_OBLIQA       = 20       # same on ObliQA (the released ObliQA notebooks use 50; both are reported in the paper)
WINDOW_RADIUS  = 1        # +-1 neighbouring passage around each candidate
TAU_IN         = 0.0      # inner decay score floor
DELTA_IN       = 2.0      # inner decay drop tolerance of the repo code (src/pipeline.py default). It only matters
                          # when SEVERAL sentences are kept (best-of-3, query-guided, the multi-sentence variant).
BGE_NAME       = "BAAI/bge-base-en-v1.5"
CE_NAME        = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BERT_MODEL     = "roberta-large"           # BERTScore (raw F1, no baseline rescaling)
QWEN_NAME      = "Qwen/Qwen3-4B"
MISTRAL_NAME   = "mistralai/Mistral-7B-Instruct-v0.3"
NLI_NAME       = "cross-encoder/nli-deberta-v3-base"

# ---------------------------------------------------------------------------------------------
# 3. TEXT HELPERS
# ---------------------------------------------------------------------------------------------
def tokenize(text):
    """BM25 tokenizer: lowercase, strip punctuation, split on whitespace (same as the repo)."""
    return str(text).lower().translate(str.maketrans("", "", string.punctuation)).split()

def token_f1(prediction, ground_truth):
    """Set-based token F1 used throughout the paper (identical to src/evaluate.py):
    precision = |overlap of word SETS| / (number of predicted tokens), recall likewise with the reference length."""
    pred = str(prediction).lower().split(); gt = str(ground_truth).lower().split()
    common = set(pred) & set(gt)
    if not common: return 0.0
    p, r = len(common) / len(pred), len(common) / len(gt)
    return 2 * p * r / (p + r)

def inner_decay(scored, tau=TAU_IN, delta=DELTA_IN):
    """Keep sentences in descending cross-encoder score while (a) score >= tau and (b) the drop from the previous kept
    sentence is <= delta. Always keeps at least the best sentence. `scored` = list of (score, sentence)."""
    if not scored: return []
    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    kept = []
    for i, (s, t) in enumerate(scored):
        if s < tau: break
        if i > 0 and (scored[i - 1][0] - s) > delta: break
        kept.append((s, t))
    return kept if kept else [scored[0]]

def rrf_fuse(rank_lists, key_fn, k=RRF_K):
    """Reciprocal Rank Fusion over several ranked lists of indices. Scores are accumulated PER PASSAGE KEY
    (e.g. (doc, chunk_idx)), so a passage that appears several times accumulates score. Returns [(key, score)] sorted."""
    acc = {}
    for lst in rank_lists:
        for rank, idx in enumerate(lst):
            kk = key_fn(idx); acc[kk] = acc.get(kk, 0.0) + 1.0 / (k + rank + 1)
    return sorted(acc.items(), key=lambda x: x[1], reverse=True)

# ---------------------------------------------------------------------------------------------
# 4. REGULATIONQA LABELS
# ---------------------------------------------------------------------------------------------
# The released CSV stores 40 raw `document_source` strings: variants of the same instrument (e.g. "GDPR" and
# "General Data Protection Regulation (EU) 2016/679") and cross-document labels containing " + ".  This maps the
# 12 single-source instruments to ONE name each. Derived from the actual labels in regulationqa_v1.csv; the counts
# reproduce the paper (607 single-source, 395 cross-document) and step 03 asserts that.
SOURCE_CANON = [   # (substring that identifies the label, canonical instrument name)
    ("e-Governance Policy Initiatives",            "e-Governance Policy India"),
    ("General Data Protection Regulation",         "GDPR"),
    ("GDPR",                                       "GDPR"),
    ("OECD/LEGAL/0479",                            "OECD Digital Security Guidelines"),
    ("NIST SP 800-66r2",                           "NIST SP 800-66r2"),
    ("DPDP Act 2023",                              "DPDP Act 2023"),
    ("NCIIPC Guidelines",                          "NCIIPC Guidelines"),
    ("ISO 27001:2022",                             "ISO/IEC 27001:2022"),
    ("ISO/IEC 27001:2022",                         "ISO/IEC 27001:2022"),
    ("Presentation - CSCRF",                       "SEBI CSCRF"),
    ("State Data Centre security guidelines",      "State DC Security Guidelines"),
    ("Digital Personal Data Protection Rules, 2025", "DPDP Rules 2025"),
    ("PCI Security Standards",                     "PCI Security Standards"),
    ("Takshashila",                                "TRAI Telecom Advisory"),
]
def is_cross(label):
    label = str(label); return (" + " in label) or label.strip() == "Cross-Document"
def canonical_source(label):
    """Canonical single-source instrument name, or 'CROSS' for cross-document questions."""
    if is_cross(label): return "CROSS"
    for sub, name in SOURCE_CANON:
        if sub in str(label): return name
    raise ValueError(f"Unmapped document_source label: {label!r} -- add it to SOURCE_CANON in adsir_common.py")

def load_regulationqa():
    df = pd.read_csv(find("regulationqa_v1.csv"))
    df = df[[c for c in df.columns if not c.startswith("Unnamed")]].copy()
    df["source"] = df["document_source"].map(canonical_source)
    df["is_cross"] = (df["source"] == "CROSS").astype(int)
    assert len(df) == 1002 and df.is_cross.sum() == 395 and (df.is_cross == 0).sum() == 607, "RegulationQA counts differ from the paper"
    return df

# ---------------------------------------------------------------------------------------------
# 5. STATISTICS
# ---------------------------------------------------------------------------------------------
def paired_bootstrap(a, b, B=2000, seed=0):
    """Mean difference a-b with a 95% percentile bootstrap CI over questions (paired)."""
    a, b = np.asarray(a, float), np.asarray(b, float); d = a - b; n = len(d)
    rng = np.random.default_rng(seed)
    means = np.array([d[rng.integers(0, n, n)].mean() for _ in range(B)])
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def save_json(obj, name):
    p = WORK / name
    with open(p, "w") as f: json.dump(obj, f, indent=2, default=float)
    print("saved", p)

# ---------------------------------------------------------------------------------------------
# 6. REGULATIONQA RESOURCES + CACHE HELPERS  (used by steps 02-05, 08-11)
# ---------------------------------------------------------------------------------------------
def load_rqa_resources(load_models=True):
    """Loads the released RegulationQA index (chunks.pkl, faiss_index.bin, bm25.pkl) and, optionally, BGE + cross-encoder.
    Returns a dict R with: chunks, index, bm25, bge, ce, doc_list, pos_in_doc, pid_to_idx, window_text()."""
    import faiss
    with open(find("chunks.pkl"), "rb") as f: chunks = pickle.load(f)
    index = faiss.read_index(find("faiss_index.bin"))
    with open(find("bm25.pkl"), "rb") as f: bm25 = pickle.load(f)
    assert len(chunks) == index.ntotal, (len(chunks), index.ntotal)
    R = dict(chunks=chunks, index=index, bm25=bm25)
    # Chunks of one document are stored consecutively, so the +-1 window is just the neighbouring chunks in the list.
    doc_list = defaultdict(list)
    for i, c in enumerate(chunks): doc_list[c["doc_name"]].append(i)
    R["doc_list"] = doc_list
    R["pos_in_doc"] = {i: (d, p) for d, ids in doc_list.items() for p, i in enumerate(ids)}
    pid_to_idx = {}
    for i, c in enumerate(chunks): pid_to_idx.setdefault((c["doc_name"], c["chunk_idx"]), i)
    R["pid_to_idx"] = pid_to_idx
    def window_text(idx, radius=WINDOW_RADIUS):
        d, p = R["pos_in_doc"][idx]; ids = doc_list[d]
        return " ".join(chunks[ids[q]]["text"] for q in range(p - radius, p + radius + 1) if 0 <= q < len(ids))
    R["window_text"] = window_text
    if load_models:
        import torch
        from sentence_transformers import SentenceTransformer, CrossEncoder
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        R["bge"] = SentenceTransformer(BGE_NAME, device=dev)
        R["ce"] = CrossEncoder(CE_NAME, device=dev)
    print(f"RegulationQA resources: {len(chunks)} chunks, {len(doc_list)} documents")
    return R

def load_cache(prefix):
    """Concatenate all shards `<prefix>_NNNN.pkl` found in WORK (sorted by question row / index)."""
    out = []
    for p in sorted(glob.glob(str(WORK / f"{prefix}_*.pkl"))): out += pickle.load(open(p, "rb"))
    key = "row" if (out and "row" in out[0]) else "q"
    out.sort(key=lambda q: q[key]); return out

def results_for(q, tau=TAU_IN, delta=DELTA_IN, use_window=True):
    """Repo rule (src/pipeline.py). For every candidate passage: rank key = best cross-encoder score among the passage's OWN
    sentences; the text is the inner-decay-kept sentences of the +-1 window joined with spaces (or of the own passage when
    use_window=False). Returns candidates sorted by rank key."""
    S, C = q["sents"], q["ce"]; res = []
    for c in q["cands"]:
        top_ce = max(float(C[j]) for j in c["own"])
        ids = c["win"] if use_window else c["own"]
        kept = inner_decay([(float(C[j]), S[j]) for j in ids], tau, delta)
        res.append(dict(doc=c["doc"], chunk_idx=c["chunk_idx"], top_ce=top_ce, kept=kept, text=" ".join(t for _, t in kept)))
    res.sort(key=lambda r: r["top_ce"], reverse=True)
    return res

def span_modes_repo(res, gt):
    """The three reported modes under the REPO rule (joined kept sentences), for diagnostics / sensitivity (step 04)."""
    if not res: return dict(top1="", qg="", orc="")
    top3 = res[:3]; sents = [(s, t) for r in top3 for s, t in r["kept"]]
    return dict(top1=res[0]["text"],
                qg=max(sents, key=lambda x: x[0])[1],                    # highest cross-encoder sentence over the top-3 candidates
                orc=max(sents, key=lambda x: token_f1(x[1], gt))[1])     # oracle: best token-F1 sentence (uses the reference!)

def span_modes_saved_run(q, gt, n_top=3):
    """The rule that REPRODUCES the saved submitted run (about 88% of top-1 spans identical; see README, 'honest note').
    Candidates ranked by own-passage max CE score; each candidate contributes ONE span = the best-scoring sentence of its
    +-1 window. top-1 = best sentence of candidate 1. query-guided = highest-scoring of the three. oracle = best token-F1 of the three."""
    S, C = q["sents"], q["ce"]
    if not q["cands"]: return dict(top1="", qg="", orc="")
    top = sorted(q["cands"], key=lambda c: max(float(C[j]) for j in c["own"]), reverse=True)[:n_top]
    best = [max(c["win"], key=lambda j: float(C[j])) for c in top]
    return dict(top1=S[best[0]],
                qg=S[max(best, key=lambda j: float(C[j]))],
                orc=max((S[j] for j in best), key=lambda s: token_f1(s, gt)))

# STEP 02 - RegulationQA: run retrieval ONCE and cache everything
**What this does:** for each of the 1,002 questions it runs the retrieval pipeline up to cross-encoder scoring and stores the
result. Every later RegulationQA step (03, 04, 05, 08, 09, 11) works from this cache, so the slow GPU part is done once.

**Pipeline per question** (paper Section "Method"):
1. BGE (FAISS inner product on normalised vectors) top-100 and BM25 top-100.
2. Reciprocal Rank Fusion, k = 60, accumulated per passage (doc, chunk_idx); keep the top m = 20 fused passages.
3. For each passage: sentences of the passage itself ("own") and of its +-1 neighbour window ("win") via NLTK.
4. Cross-encoder (ms-marco-MiniLM-L-6-v2) scores every unique sentence against the question.

**Cache content per question:** `sents` (unique sentences), `ce` (their scores), `cands` (per passage: own / window sentence ids),
`bge10`, `bm10` (top-10 chunk ids of each single retriever, used for the full-chunk baselines).

**Run on:** Kaggle, GPU T4, Internet ON. **Time:** ~25-35 min. Resumable (existing shards are skipped).
**Writes:** `rqa_cache_0000.pkl ... rqa_cache_1000.pkl` (11 shards, in the output folder)

In [3]:
!pip install -q faiss-cpu rank-bm25 sentence-transformers nltk
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)

CACHE_PREFIX = "rqa_cache"
M = M_RQA                      # candidate pool size; change to 50 to test the repo default (writes under another prefix!)
if M != 20: CACHE_PREFIX = f"rqa_cache_m{M}"

R = load_rqa_resources(load_models=True)
chunks, index, bm25, bge, ce = R["chunks"], R["index"], R["bm25"], R["bge"], R["ce"]
pid_to_idx, window_text = R["pid_to_idx"], R["window_text"]
rqa = load_regulationqa()
print("questions:", len(rqa))

[find] chunks.pkl                       -> /kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_artifacts/chunks.pkl
[find] faiss_index.bin                  -> /kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_artifacts/faiss_index.bin
[find] bm25.pkl                         -> /kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_artifacts/bm25.pkl


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (13 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

RegulationQA resources: 766 chunks, 20 documents
[find] regulationqa_v1.csv              -> /kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_artifacts/regulationqa_v1.csv
questions: 1002


### Retrieval + scoring of one question

In [4]:
def rrf_top(query):
    qe = bge.encode([query], normalize_embeddings=True).astype(np.float32)
    _, bi = index.search(qe, PRE_K); bi = bi[0]                              # dense top-100
    bm_i = np.argsort(bm25.get_scores(tokenize(query)))[::-1][:PRE_K]        # BM25 top-100
    fused = rrf_fuse([list(bi), list(bm_i)], key_fn=lambda i: (chunks[i]["doc_name"], chunks[i]["chunk_idx"]))
    return fused[:M], [int(x) for x in bi[:10]], [int(x) for x in bm_i[:10]]

def process(query):
    top, bge10, bm10 = rrf_top(query)
    sents, uniq, cands, seen = [], {}, [], set()
    def sid(s):                                    # give every distinct sentence one id, so the CE scores it only once
        if s not in uniq: uniq[s] = len(sents); sents.append(s)
        return uniq[s]
    for pid, rrf_score in top:
        if pid in seen: continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None: continue
        c = chunks[idx]
        own = sent_tokenize(c["text"])
        if not own: continue
        win = sent_tokenize(window_text(idx))
        cands.append(dict(doc=c["doc_name"], chunk_idx=c["chunk_idx"], idx=idx, rrf=float(rrf_score),
                          own=[sid(s) for s in own], win=[sid(s) for s in win]))
    scores = ce.predict([[query, s] for s in sents], batch_size=128, show_progress_bar=False) if sents else []
    return dict(sents=sents, ce=np.asarray(scores, dtype=np.float32), cands=cands, bge10=bge10, bm10=bm10)

### Run all 1,002 questions in shards of 100 (safe to stop and re-run)

In [5]:
SHARD, t0 = 100, time.time()
for start in range(0, len(rqa), SHARD):
    path = WORK / f"{CACHE_PREFIX}_{start:04d}.pkl"
    if path.exists(): print("skip", path.name); continue
    out = []
    for i in range(start, min(start + SHARD, len(rqa))):
        r = process(str(rqa.iloc[i]["question"])); r["row"] = i; out.append(r)
    pickle.dump(out, open(path, "wb"))
    print(f"shard {start:4d}-{start+len(out)-1:4d} saved | {time.time()-t0:6.0f}s | avg unique sentences/question {np.mean([len(o['sents']) for o in out]):.0f}")
print("shards on disk:", len(glob.glob(str(WORK / f'{CACHE_PREFIX}_*.pkl'))), "(expect 11)")

shard    0-  99 saved |    109s | avg unique sentences/question 793
shard  100- 199 saved |    224s | avg unique sentences/question 764
shard  200- 299 saved |    326s | avg unique sentences/question 733
shard  300- 399 saved |    432s | avg unique sentences/question 831
shard  400- 499 saved |    550s | avg unique sentences/question 846
shard  500- 599 saved |    670s | avg unique sentences/question 972
shard  600- 699 saved |    780s | avg unique sentences/question 799
shard  700- 799 saved |    889s | avg unique sentences/question 634
shard  800- 899 saved |    996s | avg unique sentences/question 675
shard  900- 999 saved |   1117s | avg unique sentences/question 801
shard 1000-1001 saved |   1119s | avg unique sentences/question 1032
shards on disk: 11 (expect 11)


# STEP 12 - Controlled study: single-sentence vs adaptive multi-sentence span extraction
**Question this answers:** does the adaptive score-drop rule (tau_in, delta_in) beat simply returning the best sentence? And is the
effect of *candidate selection* separable from the effect of *sentence selection*?

**Audit of the paper (what was and was not compared):**
* **Table 3 (delta sensitivity)** scores the *candidate-level multi-sentence text* (all inner-decay sentences of a candidate's
  +-1 window, joined in score order) and takes the oracle best of the top-3 candidates. Its "Words" column is 69-202.
* **Table 5 (main results)** uses *one sentence* per candidate (the best-scoring sentence of the window). Spans are about one sentence long.
* So Tables 3 and 5 do NOT evaluate comparable configurations. The paper never runs top-1 single sentence against top-1 adaptive
  multi-sentence on the same questions, and the best oracle value in Table 3 (0.3698) is below the single-sentence oracle of
  Table 5 (0.3737). The data in the paper therefore does not establish that adaptive multi-sentence extraction is better.
* Candidate selection and sentence selection are not separated anywhere. This step separates them with sentence-level ceilings.

**Configurations (all on the same 1,002 questions, same retrieval, same windows, same cached cross-encoder scores)**
| id | what | uses reference? |
|---|---|---|
| A  | top-1 single sentence (best sentence of the rank-1 candidate's window) | no |
| B  | top-1 adaptive multi-sentence (inner decay on the rank-1 window) | no |
| C  | query-guided single sentence (best sentence over the top-3 windows) | no |
| D  | query-guided adaptive multi-sentence (inner decay on the pooled top-3 windows) - NEW variant, not a paper result | no |
| E1 | ORACLE best-of-3, one best sentence per candidate (= paper Table 5 best-of-3) | **YES** |
| E2 | ORACLE best sentence anywhere in the top-3 windows | **YES** |
| E3 | ORACLE best-of-3 candidate-level adaptive text (= paper Table 3 definition) | **YES** |

B and D are produced in two assemblies: `score` order (the paper/repo rule: sentences concatenated in descending cross-encoder
score) and `doc` order (reading order: candidate rank, then position in the window). Reading order is more defensible for a
human reader; the score order is kept as the original method. Sentences from different documents have no shared reading order,
so for D the order is by candidate rank and then position.

**Hyperparameters:** tau_in = 0, delta_in in {0.5, 1, 1.5, 2, 2.5, 3, 5, 5.5, 20}. delta (and optionally tau) is **chosen on a
validation half** and the chosen value is reported on the **test half**; full-set numbers are descriptive only.

**Honest limits:** exact match is near zero for extractive spans (reported for completeness). "Completeness" is a proxy: recall of
the reference's tokens and of its condition terms (shall/must/unless/if/within, numbers); it does not prove legal completeness.
Judge results (Qwen/Mistral) are NOT recomputed here; they stay in step 08. Runtime counts only selection from the cache,
not retrieval or cross-encoder scoring.

**Run on:** CPU (~5 min); BERTScore needs a GPU (set RUN_BERTSCORE). Needs the step-02 cache. **Writes:** folder `results_span_study/`.

In [6]:
!pip install -q bert-score
import re, importlib, platform, itertools
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

RUN_BERTSCORE = True            # needs GPU + `pip install bert-score`; set False to skip
RUN_TAU_GRID  = True            # small tau x delta grid on the validation half only
SEED          = 42
DELTAS        = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0, 5.5, 20.0]
OUT = WORK / "results_span_study"; OUT.mkdir(exist_ok=True)
t_start = time.time()

cache = load_cache("rqa_cache"); rqa = load_regulationqa()
assert len(cache) == 1002, f"cache has {len(cache)} questions - run step 02 first"
GT = [str(g) for g in rqa["ground_truth_answer"]]
ROWS = [q["row"] for q in cache]; assert ROWS == list(range(1002))

[find] regulationqa_v1.csv              -> /kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_artifacts/regulationqa_v1.csv


### Split: 50% validation (choose thresholds) / 50% test (report), stratified by instrument, fixed seed

In [7]:
rng = np.random.default_rng(SEED)
split = np.array(["test"] * len(rqa), dtype=object)
for src, idx in rqa.groupby("source").indices.items():
    idx = np.array(idx); rng.shuffle(idx); split[idx[: len(idx) // 2]] = "val"
rqa["split"] = split
print(rqa.split.value_counts().to_dict(), "| cross-document in val/test:", rqa.groupby("split").is_cross.sum().to_dict())
VAL = np.where(split == "val")[0]; TEST = np.where(split == "test")[0]

{'test': 503, 'val': 499} | cross-document in val/test: {'test': 198, 'val': 197}


### Selection rules (the only place where the configurations are defined)

In [8]:
def ranked(q):
    """Candidates ranked by the best cross-encoder score of their OWN passage (the paper's ranking rule)."""
    C = q["ce"]; return sorted(q["cands"], key=lambda c: max(float(C[j]) for j in c["own"]), reverse=True)

def pool(q, n_cands):
    """Unique sentences of the +-1 windows of the top n_cands candidates: dicts(score, text, rank, pos). Order of first appearance."""
    seen, items = set(), []
    for r, c in enumerate(ranked(q)[:n_cands]):
        for p, j in enumerate(c["win"]):
            if j in seen: continue
            seen.add(j); items.append(dict(score=float(q["ce"][j]), text=q["sents"][j], rank=r, pos=p, sid=j))
    return items

def single(items):                       # the highest-scoring sentence (A, C)
    return max(items, key=lambda x: x["score"])["text"] if items else ""

def adaptive(items, tau, delta, order):  # inner decay (B, D); order = 'score' (paper) or 'doc' (reading order)
    if not items: return ""
    by_text = {it["text"]: it for it in items}
    kept = inner_decay([(it["score"], it["text"]) for it in items], tau, delta)   # sorted by score, always >= 1 sentence
    objs = [by_text[t] for _, t in kept]
    if order == "doc": objs.sort(key=lambda x: (x["rank"], x["pos"]))
    return " ".join(o["text"] for o in objs)

def cfg_A(q):             return single(pool(q, 1))
def cfg_B(q, tau, d, o):  return adaptive(pool(q, 1), tau, d, o)
def cfg_C(q):             return single(pool(q, 3))
def cfg_D(q, tau, d, o):  return adaptive(pool(q, 3), tau, d, o)
def cand_items(q, r):
    """Full +-1 window of the r-th ranked candidate (NOT de-duplicated against other windows), as dicts(score, text, rank, pos)."""
    cs = ranked(q)
    if r >= len(cs): return []
    return [dict(score=float(q["ce"][j]), text=q["sents"][j], rank=r, pos=p, sid=j) for p, j in enumerate(cs[r]["win"])]
def cfg_E1(q, gt):        # ORACLE: best sentence of each of the 3 candidate windows, then the one with best token F1 (paper Table 5 best-of-3)
    c3 = [single(cand_items(q, r)) for r in range(3) if cand_items(q, r)]
    return max(c3, key=lambda t: token_f1(t, gt)) if c3 else ""
def cfg_E2(q, gt):        # ORACLE: best token-F1 sentence anywhere in the top-3 windows
    items = pool(q, 3); return max(items, key=lambda it: token_f1(it["text"], gt))["text"] if items else ""
def cfg_E3(q, gt, tau, d):# ORACLE: per candidate the adaptive joined text (score order); best token F1 among the top 3 (paper Table 3 definition)
    texts = [adaptive(cand_items(q, r), tau, d, "score") for r in range(3) if cand_items(q, r)]
    return max(texts, key=lambda t: token_f1(t, gt)) if texts else ""

### Metrics per output

In [9]:
COND = re.compile(r"\b(shall|must|may not|may|unless|if|provided|except|subject to|within|only|not|no|any|all|each|every|before|after|until)\b|\d+")
def norm(t): return " ".join(re.sub(r"[^\w\s]", " ", str(t).lower()).split())
def cond_terms(t): return [m.group(0) for m in COND.finditer(str(t).lower())]
def metrics_row(txt, gt):
    w = str(txt).split(); g = str(gt).lower().split(); gset = set(g); pset = set(str(txt).lower().split())
    ct = cond_terms(gt)
    return dict(f1=token_f1(txt, gt), em=float(norm(txt) == norm(gt)), words=len(w), empty=float(len(w) == 0),
                ref_recall=(len(gset & pset) / len(gset)) if gset else 0.0,
                cond_recall=(sum(c in set(cond_terms(txt)) for c in ct) / len(ct)) if ct else np.nan)

### Run every configuration on every question (texts kept for the main configs only)

In [10]:
TAU0 = 0.0
configs = {}                                   # name -> list[str] outputs, aligned with questions
configs["A_top1_single"] = [cfg_A(q) for q in cache]
configs["C_qg_single"]   = [cfg_C(q) for q in cache]
configs["E1_ORACLE_best3_sentence_per_candidate"] = [cfg_E1(q, GT[q["row"]]) for q in cache]
configs["E2_ORACLE_best_sentence_in_top3_windows"] = [cfg_E2(q, GT[q["row"]]) for q in cache]
for d in DELTAS:
    for o in ("score", "doc"):
        configs[f"B_top1_adaptive_d{d}_{o}"] = [cfg_B(q, TAU0, d, o) for q in cache]
        configs[f"D_qg_adaptive_d{d}_{o}"]   = [cfg_D(q, TAU0, d, o) for q in cache]
    configs[f"E3_ORACLE_best3_candidate_adaptive_d{d}"] = [cfg_E3(q, GT[q["row"]], TAU0, d) for q in cache]
print(len(configs), "configurations x", len(cache), "questions")

recs = []
for name, outs in configs.items():
    for i, (t, g) in enumerate(zip(outs, GT)): recs.append(dict(row=i, split=split[i], config=name, **metrics_row(t, g)))
M = pd.DataFrame(recs)

49 configurations x 1002 questions


### Runtime (selection from the cache only), measured per configuration family

In [11]:
def timeit(fn):
    t0 = time.perf_counter(); [fn(q) for q in cache]; return time.perf_counter() - t0
runtime = {"A_top1_single": timeit(cfg_A), "B_top1_adaptive_d2.0_score": timeit(lambda q: cfg_B(q, 0.0, 2.0, "score")),
           "C_qg_single": timeit(cfg_C), "D_qg_adaptive_d2.0_score": timeit(lambda q: cfg_D(q, 0.0, 2.0, "score"))}
print({k: f"{v:.2f}s total, {1000*v/len(cache):.2f} ms/question" for k, v in runtime.items()})

{'A_top1_single': '0.12s total, 0.12 ms/question', 'B_top1_adaptive_d2.0_score': '0.13s total, 0.13 ms/question', 'C_qg_single': '0.16s total, 0.16 ms/question', 'D_qg_adaptive_d2.0_score': '0.21s total, 0.21 ms/question'}


### Choose delta on the VALIDATION half (never on test), then report on the TEST half

In [12]:
def f1_on(name, idx): return M[(M.config == name) & (M.row.isin(idx))].f1.mean()
def best_delta(prefix, order):
    scores = {d: f1_on(f"{prefix}_d{d}_{order}", VAL) for d in DELTAS}
    mx = max(scores.values()); return min(d for d, v in scores.items() if v >= mx - 1e-12), scores     # ties -> smaller delta (shorter span)
chosen = {}
for prefix in ("B_top1_adaptive", "D_qg_adaptive"):
    for o in ("score", "doc"):
        d, sc = best_delta(prefix, o); chosen[f"{prefix}_{o}"] = d
        print(f"{prefix:16s} {o:5s} validation-best delta = {d}   (val F1 by delta: " + ", ".join(f"{k}:{v:.4f}" for k, v in sc.items()) + ")")

if RUN_TAU_GRID:
    print("\nBroader check on VALIDATION only: tau x delta for B (score order)")
    grid = {}
    for tau in (-2.0, -1.0, 0.0, 1.0, 2.0):
        for d in (1.0, 2.0, 5.0):
            outs = [cfg_B(q, tau, d, "score") for q in cache]
            grid[(tau, d)] = np.mean([token_f1(outs[i], GT[i]) for i in VAL])
    print(pd.Series(grid).unstack().round(4).to_string())
    pd.Series(grid).unstack().to_csv(OUT / "tau_delta_grid_validation.csv")

B_top1_adaptive  score validation-best delta = 0.5   (val F1 by delta: 0.5:0.2942, 1.0:0.2900, 1.5:0.2729, 2.0:0.2626, 2.5:0.2491, 3.0:0.2395, 5.0:0.1961, 5.5:0.1918, 20.0:0.1813)
B_top1_adaptive  doc   validation-best delta = 0.5   (val F1 by delta: 0.5:0.2942, 1.0:0.2900, 1.5:0.2729, 2.0:0.2626, 2.5:0.2491, 3.0:0.2395, 5.0:0.1961, 5.5:0.1918, 20.0:0.1813)
D_qg_adaptive    score validation-best delta = 0.5   (val F1 by delta: 0.5:0.2910, 1.0:0.2731, 1.5:0.2440, 2.0:0.2243, 2.5:0.2110, 3.0:0.1987, 5.0:0.1534, 5.5:0.1516, 20.0:0.1398)
D_qg_adaptive    doc   validation-best delta = 0.5   (val F1 by delta: 0.5:0.2910, 1.0:0.2731, 1.5:0.2440, 2.0:0.2243, 2.5:0.2110, 3.0:0.1987, 5.0:0.1534, 5.5:0.1516, 20.0:0.1398)

Broader check on VALIDATION only: tau x delta for B (score order)
         1.0     2.0     5.0
-2.0  0.2884  0.2574  0.1770
-1.0  0.2893  0.2603  0.1865
 0.0  0.2900  0.2626  0.1961
 1.0  0.2905  0.2649  0.2061
 2.0  0.2914  0.2700  0.2204


### Which configurations go into the headline table

In [13]:
dB, dD = chosen["B_top1_adaptive_score"], chosen["D_qg_adaptive_score"]
MAIN = ["A_top1_single", f"B_top1_adaptive_d2.0_score", f"B_top1_adaptive_d{dB}_score", f"B_top1_adaptive_d{dB}_doc",
        "C_qg_single", f"D_qg_adaptive_d2.0_score", f"D_qg_adaptive_d{dD}_score", f"D_qg_adaptive_d{dD}_doc",
        "E1_ORACLE_best3_sentence_per_candidate", "E2_ORACLE_best_sentence_in_top3_windows", "E3_ORACLE_best3_candidate_adaptive_d2.0"]
MAIN = list(dict.fromkeys(MAIN))

### BERTScore (main configurations only)

In [14]:
bert = {}
if RUN_BERTSCORE:
    try:
        from bert_score import score as bscore
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        for name in MAIN:
            cands_ = [t if str(t).strip() else " " for t in configs[name]]
            _, _, F = bscore(cands_, GT, model_type=BERT_MODEL, lang="en", batch_size=32, device=dev, verbose=False)
            bert[name] = F.numpy(); print(f"BERTScore {name:48s} {F.mean().item():.4f}")
    except Exception as e:
        print("BERTScore skipped:", repr(e))
for name, arr in bert.items():
    M.loc[M.config == name, "bertscore"] = arr[M.loc[M.config == name, "row"].values]

BERTScore skipped: ModuleNotFoundError("No module named 'bert_score'")


### Summary tables (all / validation / test)

In [15]:
def summarize(df):
    g = df.groupby("config")
    out = pd.DataFrame(dict(n=g.size(), F1=g.f1.mean(), EM=g.em.mean(), ref_recall=g.ref_recall.mean(), cond_recall=g.cond_recall.mean(),
                            words_mean=g.words.mean(), words_median=g.words.median(), words_std=g.words.std(), empty_pct=100 * g.empty.mean()))
    if "bertscore" in df: out["BERTScore_F1"] = g.bertscore.mean()
    return out
summaries = {}
for nm, sel in [("all", np.arange(1002)), ("val", VAL), ("test", TEST)]:
    s = summarize(M[M.row.isin(sel)]); summaries[nm] = s
    s.loc[[c for c in MAIN if c in s.index]].round(4).to_csv(OUT / f"summary_{nm}.csv")
M[M.config.isin(MAIN)].assign(text=lambda d: [configs[c][r] for c, r in zip(d.config, d.row)]).to_csv(OUT / "per_question_main_configs.csv", index=False)
M.drop(columns=[]).to_csv(OUT / "per_question_all_configs_metrics.csv", index=False)
print("\n=== TEST half (thresholds chosen on validation) ===")
print(summaries["test"].loc[MAIN, ["F1", "ref_recall", "cond_recall", "words_mean", "words_median", "empty_pct"] + (["BERTScore_F1"] if bert else [])].round(4).to_string())


=== TEST half (thresholds chosen on validation) ===
                                             F1  ref_recall  cond_recall  words_mean  words_median  empty_pct
config                                                                                                       
A_top1_single                            0.2888      0.4184       0.4306     57.8708          37.0        0.0
B_top1_adaptive_d2.0_score               0.2429      0.4702       0.4915    133.4076          53.0        0.0
B_top1_adaptive_d0.5_score               0.2801      0.4354       0.4443     68.6859          40.0        0.0
B_top1_adaptive_d0.5_doc                 0.2801      0.4354       0.4443     68.6859          40.0        0.0
C_qg_single                              0.2897      0.4179       0.4341     54.1630          37.0        0.0
D_qg_adaptive_d2.0_score                 0.2195      0.5209       0.5684    266.3837          71.0        0.0
D_qg_adaptive_d0.5_score                 0.2717      0.4543       0

### Hyperparameter sensitivity CSV (val / test / all) and the candidate-vs-sentence decomposition

In [16]:
sens = []
for prefix in ("B_top1_adaptive", "D_qg_adaptive", "E3_ORACLE_best3_candidate_adaptive"):
    for o in (("score", "doc") if not prefix.startswith("E3") else ("score",)):
        for d in DELTAS:
            name = f"{prefix}_d{d}_{o}" if not prefix.startswith("E3") else f"{prefix}_d{d}"
            sens.append(dict(family=prefix, order=o, delta=d, F1_val=f1_on(name, VAL), F1_test=f1_on(name, TEST), F1_all=f1_on(name, np.arange(1002)),
                             words_mean=M[M.config == name].words.mean(), words_median=M[M.config == name].words.median()))
sens = pd.DataFrame(sens); sens.to_csv(OUT / "hyperparameter_sensitivity.csv", index=False)

# Separate candidate selection from sentence selection with sentence-level ceilings (uses the reference, diagnostic only)
def ceiling(q, gt, n_cands):
    items = pool(q, n_cands); return max((token_f1(it["text"], gt) for it in items), default=0.0)
dec = pd.DataFrame(dict(ceil_top1_window=[ceiling(q, GT[q["row"]], 1) for q in cache], ceil_top3_windows=[ceiling(q, GT[q["row"]], 3) for q in cache],
                        ceil_all20_windows=[ceiling(q, GT[q["row"]], 20) for q in cache]))
dec["A_actual"] = M[M.config == "A_top1_single"].sort_values("row").f1.values
D_ = dec.mean().round(4).to_dict()
decomp = dict(ceiling_all_20_candidates=D_["ceil_all20_windows"], ceiling_top1_window=D_["ceil_top1_window"], A_top1_single_actual=D_["A_actual"],
              candidate_selection_loss=round(D_["ceil_all20_windows"] - D_["ceil_top1_window"], 4),
              sentence_selection_loss_top1=round(D_["ceil_top1_window"] - D_["A_actual"], 4))
print("\nDecomposition (token F1, ORACLE ceilings are diagnostics, not methods):"); [print(f"  {k:34s} {v}") for k, v in decomp.items()]
save_json(decomp, "results_span_study/decomposition.json")


Decomposition (token F1, ORACLE ceilings are diagnostics, not methods):
  ceiling_all_20_candidates          0.4618
  ceiling_top1_window                0.3966
  A_top1_single_actual               0.2961
  candidate_selection_loss           0.0652
  sentence_selection_loss_top1       0.1005
saved /kaggle/working/results_span_study/decomposition.json


### Paired bootstrap comparisons (TEST half and full set)

In [17]:
def vec(name, idx): return M[(M.config == name) & (M.row.isin(idx))].sort_values("row").f1.values
pairs = [("B(d=2.0) - A", "B_top1_adaptive_d2.0_score", "A_top1_single"), (f"B(d={dB}, val-chosen) - A", f"B_top1_adaptive_d{dB}_score", "A_top1_single"),
         (f"B doc-order - B score-order (d={dB})", f"B_top1_adaptive_d{dB}_doc", f"B_top1_adaptive_d{dB}_score"),
         ("C - A", "C_qg_single", "A_top1_single"), ("D(d=2.0) - C", "D_qg_adaptive_d2.0_score", "C_qg_single"),
         (f"D(d={dD}, val-chosen) - C", f"D_qg_adaptive_d{dD}_score", "C_qg_single"), ("E1 - E3(d=2.0) [oracle vs oracle]", "E1_ORACLE_best3_sentence_per_candidate", "E3_ORACLE_best3_candidate_adaptive_d2.0")]
rows = []
for lab, a, b in pairs:
    for nm, idx in [("test", TEST), ("all", np.arange(1002))]:
        m, lo, hi = paired_bootstrap(vec(a, idx), vec(b, idx)); rows.append(dict(comparison=lab, subset=nm, mean_diff_F1=m, ci_lo=lo, ci_hi=hi, ci_excludes_0=bool(lo > 0 or hi < 0)))
BS = pd.DataFrame(rows); BS.round(4).to_csv(OUT / "paired_bootstrap.csv", index=False); print("\n", BS.round(4).to_string(index=False))


                          comparison subset  mean_diff_F1   ci_lo   ci_hi  ci_excludes_0
                       B(d=2.0) - A   test       -0.0458 -0.0578 -0.0345           True
                       B(d=2.0) - A    all       -0.0434 -0.0515 -0.0349           True
           B(d=0.5, val-chosen) - A   test       -0.0087 -0.0136 -0.0042           True
           B(d=0.5, val-chosen) - A    all       -0.0090 -0.0126 -0.0056           True
B doc-order - B score-order (d=0.5)   test        0.0000  0.0000  0.0000          False
B doc-order - B score-order (d=0.5)    all        0.0000  0.0000  0.0000          False
                              C - A   test        0.0010 -0.0006  0.0027          False
                              C - A    all        0.0022  0.0007  0.0038           True
                       D(d=2.0) - C   test       -0.0703 -0.0849 -0.0560           True
                       D(d=2.0) - C    all       -0.0764 -0.0878 -0.0652           True
           D(d=0.5, val-chosen

### Plots

In [18]:
fig, ax = plt.subplots(figsize=(6, 4))
for fam, lab in [("B_top1_adaptive", "B top-1 adaptive"), ("D_qg_adaptive", "D query-guided adaptive")]:
    s = sens[(sens.family == fam) & (sens.order == "score")]; ax.plot(s.delta, s.F1_all, marker="o", label=f"{lab} (all)")
ax.axhline(M[(M.config == "A_top1_single")].f1.mean(), ls="--", c="C2", label="A top-1 single"); ax.axhline(M[(M.config == "C_qg_single")].f1.mean(), ls=":", c="C3", label="C query-guided single")
ax.set_xscale("log"); ax.set_xlabel(r"$\delta_{in}$"); ax.set_ylabel("token F1"); ax.legend(fontsize=8); ax.set_title("delta vs token F1 (tau=0)"); fig.tight_layout(); fig.savefig(OUT / "plot_delta_vs_f1.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
for fam, lab in [("B_top1_adaptive", "B top-1 adaptive"), ("D_qg_adaptive", "D query-guided adaptive")]:
    s = sens[(sens.family == fam) & (sens.order == "score")]; ax.plot(s.delta, s.words_mean, marker="o", label=f"{lab} mean")
    ax.plot(s.delta, s.words_median, marker="x", ls="--", label=f"{lab} median")
ax.set_xscale("log"); ax.set_xlabel(r"$\delta_{in}$"); ax.set_ylabel("words"); ax.legend(fontsize=8); ax.set_title("delta vs span length"); fig.tight_layout(); fig.savefig(OUT / "plot_delta_vs_length.png", dpi=200); plt.close(fig)

S = summaries["all"].loc[MAIN]; short = lambda s: s.replace("_ORACLE", "*").replace("_adaptive", "").replace("_score", "").replace("top1_", "").replace("best3_", "")[:26]
fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(len(S)); w = 0.4
ax.bar(x - w / 2, S.F1, w, label="token F1")
if "BERTScore_F1" in S and S.BERTScore_F1.notna().any(): ax.bar(x + w / 2, S.BERTScore_F1, w, label="BERTScore F1")
ax.set_xticks(x); ax.set_xticklabels([short(i) for i in S.index], rotation=60, ha="right", fontsize=7); ax.legend(); ax.set_title("Token F1 vs BERTScore (* = oracle)"); fig.tight_layout(); fig.savefig(OUT / "plot_f1_vs_bertscore.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4.5))
for name, r in S.iterrows(): ax.scatter(r.words_mean, r.F1, marker="*" if "ORACLE" in name else "o", s=70 if "ORACLE" in name else 35); ax.annotate(short(name), (r.words_mean, r.F1), fontsize=6)
ax.set_xlabel("mean words"); ax.set_ylabel("token F1"); ax.set_title("Length vs quality (* = oracle)"); fig.tight_layout(); fig.savefig(OUT / "plot_length_vs_quality.png", dpi=200); plt.close(fig)

### Qualitative report: where adaptive selection helps, hurts, or changes nothing (B at delta=2.0 vs A; D vs C)

In [19]:
def qual(name_new, name_old, title, k=4):
    a, b = np.array(configs[name_new], dtype=object), np.array(configs[name_old], dtype=object)
    fa = np.array([token_f1(t, g) for t, g in zip(a, GT)]); fb = np.array([token_f1(t, g) for t, g in zip(b, GT)]); d = fa - fb
    out = [f"## {title}\n", f"changed output: {(a != b).mean():.1%} | better: {(d > 1e-9).mean():.1%} | worse: {(d < -1e-9).mean():.1%} | identical F1: {(abs(d) <= 1e-9).mean():.1%}\n"]
    for lab, idxs in [("Improved", np.argsort(-d)[:k]), ("Worsened", np.argsort(d)[:k]), ("Unchanged (output identical)", np.where(a == b)[0][:k])]:
        out.append(f"### {lab}\n")
        for i in idxs: out.append(f"- **Q{i} [{rqa.source.iloc[i]}]** {rqa.question.iloc[i]}\n  - reference: {GT[i][:300]}\n  - single: {b[i][:300]}  (F1 {fb[i]:.2f})\n  - adaptive: {a[i][:400]}  (F1 {fa[i]:.2f})\n")
    return "\n".join(out)
report = "# Qualitative examples (token F1, thresholds tau=0, delta=2.0)\n\n" + qual("B_top1_adaptive_d2.0_score", "A_top1_single", "B adaptive vs A single (top-1)") + "\n\n" + qual("D_qg_adaptive_d2.0_score", "C_qg_single", "D adaptive vs C single (query-guided)")
(OUT / "qualitative_report.md").write_text(report, encoding="utf-8"); print("saved qualitative_report.md")

saved qualitative_report.md


### Reproducibility log

In [20]:
def ver(p):
    try: return importlib.metadata.version(p)
    except Exception: return None
log = dict(python=platform.python_version(), platform=platform.platform(), packages={p: ver(p) for p in ["numpy", "pandas", "torch", "transformers", "sentence-transformers", "bert-score", "nltk", "faiss-cpu", "rank-bm25", "scikit-learn", "matplotlib"]},
           models=dict(dense=BGE_NAME, cross_encoder=CE_NAME, bertscore=BERT_MODEL), pipeline=dict(rrf_k=RRF_K, pre_k=PRE_K, pool_m=M_RQA, window=WINDOW_RADIUS, tau=TAU0, deltas=DELTAS),
           split=dict(seed=SEED, val=int(len(VAL)), test=int(len(TEST)), stratified_by="instrument"), chosen_delta=chosen, runtime_seconds=runtime,
           total_execution_seconds=round(time.time() - t_start, 1), cache_questions=len(cache), n_configurations=len(configs), bertscore_run=bool(bert))
save_json(log, "results_span_study/reproducibility_log.json")
print("\nAll outputs in", OUT)

saved /kaggle/working/results_span_study/reproducibility_log.json

All outputs in /kaggle/working/results_span_study


In [21]:
if not RUN_JUDGES:
    import sys; sys.exit('Judges skipped (RUN_JUDGES=False). Step 12 results are in /kaggle/working/results_span_study')

# STEP 13 - LLM judges (Qwen3-4B, Mistral-7B) on the ADAPTIVE spans from step 12  (fills the judge columns of the new Table 5 rows)
**What this does:** judges the spans of the configurations selected below, with exactly the same prompt and scoring as step 08
(0 / 1 / 2, greedy, first digit). Reports % correct-or-partial and mean score per configuration and per judge.
**Default:** only the three NEW rows (B, D, E3 at delta = 2.0). Add A, C, E1 if you want the single-sentence rows re-judged on this
same re-implementation (otherwise keep the paper's original-run values for them and footnote the provenance).
**Cost:** 1,002 spans per configuration per judge, about 20-30 min each on a T4. Set JUDGE_CONFIGS to fewer configurations if short of time.
**Needs:** step 12 output `results_span_study/per_question_main_configs.csv`. **Writes:** `results_span_study/judge_adaptive.json`, `judge_adaptive_scores.csv`.

In [22]:
!pip install -q transformers accelerate bitsandbytes scikit-learn
import torch, gc
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import cohen_kappa_score
OUT = WORK / "results_span_study"
JUDGE_CONFIGS = ["B_top1_adaptive_d2.0_score", "D_qg_adaptive_d2.0_score", "E3_ORACLE_best3_candidate_adaptive_d2.0"]
# optional: "A_top1_single", "C_qg_single", "E1_ORACLE_best3_sentence_per_candidate"
main = pd.read_csv(OUT / "per_question_main_configs.csv"); rqa = load_regulationqa()
missing = [c for c in JUDGE_CONFIGS if c not in set(main.config)]; assert not missing, f"not in step-12 main configs: {missing}"
spans = {c: main[main.config == c].sort_values("row").text.fillna("").tolist() for c in JUDGE_CONFIGS}
Q = rqa["question"].astype(str).tolist(); G = rqa["ground_truth_answer"].astype(str).tolist()
assert all(len(v) == 1002 for v in spans.values())

LLM_PROMPT = """You are evaluating a regulatory information retrieval system.

QUESTION: {question}

REFERENCE ANSWER: {ground_truth}

EXTRACTED SPAN: {extracted_span}

Does the extracted span correctly answer the question?
Consider it correct even if the wording differs from the reference,
as long as the information is accurate and relevant.

Respond with exactly one of:
2 - Fully correct: span contains the complete answer
1 - Partially correct: span contains relevant but incomplete information
0 - Incorrect: span does not answer the question

Your response (single digit only):"""

def score_one(model, tok, device, question, gt, span):
    if not str(span).strip(): return 0
    inp = tok(LLM_PROMPT.format(question=question, ground_truth=gt, extracted_span=span), return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad(): out = model.generate(**inp, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
    reply = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    del inp, out; torch.cuda.empty_cache(); gc.collect()
    return next((int(ch) for ch in reply if ch in "012"), 0)

def run_judge(model, tok, device, label):
    res = {}
    for name, sp in spans.items():
        a = np.array([score_one(model, tok, device, Q[i], G[i], sp[i]) for i in tqdm(range(1002), desc=f"{label}: {name}")])
        res[name] = dict(correct_partial=float((a >= 1).mean()), fully_correct=float((a == 2).mean()), mean_score=float(a.mean()), scores=a.tolist())
        print(f"{label:8s} {name:46s} correct+partial {100*res[name]['correct_partial']:5.1f}%  fully {100*res[name]['fully_correct']:5.1f}%")
    return res

[find] regulationqa_v1.csv              -> /kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_artifacts/regulationqa_v1.csv


### Judge 1: Qwen3-4B (fp16)

In [28]:
dev0 = "cuda:0"
tq = AutoTokenizer.from_pretrained(QWEN_NAME); mq = AutoModelForCausalLM.from_pretrained(QWEN_NAME, torch_dtype=torch.float16, device_map=dev0).eval()
qwen = run_judge(mq, tq, dev0, "Qwen"); del mq; gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (6 outputs left out)

Qwen: B_top1_adaptive_d2.0_score:   0%|          | 0/1002 [00:00<?, ?it/s]

Qwen     B_top1_adaptive_d2.0_score                     correct+partial  75.9%  fully  65.3%


Qwen: D_qg_adaptive_d2.0_score:   0%|          | 0/1002 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Judge 2: Mistral-7B-Instruct-v0.3 (4-bit)

In [23]:
qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tm = AutoTokenizer.from_pretrained(MISTRAL_NAME); mm = AutoModelForCausalLM.from_pretrained(MISTRAL_NAME, quantization_config=qc, device_map="cuda:0").eval()
mis = run_judge(mm, tm, mm.device, "Mistral")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (7 outputs left out)

Mistral: D_qg_adaptive_d2.0_score:   0%|          | 0/1002 [00:00<?, ?it/s]

Mistral  D_qg_adaptive_d2.0_score                       correct+partial  80.3%  fully  75.3%


Mistral: E3_ORACLE_best3_candidate_adaptive_d2.0:   0%|          | 0/1002 [00:00<?, ?it/s]

Mistral  E3_ORACLE_best3_candidate_adaptive_d2.0        correct+partial  90.1%  fully  85.6%


In [24]:
kappa = {c: float(cohen_kappa_score(qwen[c]["scores"], mis[c]["scores"])) for c in spans}
print("Cohen kappa Qwen vs Mistral per configuration:", {k: round(v, 3) for k, v in kappa.items()})
strip = lambda r: {k: {m: v for m, v in d.items() if m != "scores"} for k, d in r.items()}
save_json(dict(qwen=strip(qwen), mistral=strip(mis), kappa=kappa), "results_span_study/judge_adaptive.json")
pd.DataFrame({**{f"qwen|{c}": qwen[c]["scores"] for c in spans}, **{f"mistral|{c}": mis[c]["scores"] for c in spans}}).to_csv(OUT / "judge_adaptive_scores.csv", index=False)

NameError: name 'qwen' is not defined

In [30]:
!pip install -q bert-score
import glob, pandas as pd, torch
from bert_score import score as bscore

main = pd.read_csv("/kaggle/working/results_span_study/per_question_main_configs.csv")
ds = glob.glob("/kaggle/input/**/regulationqa_v1.csv", recursive=True)[0]
df = pd.read_csv(ds)
GT = df["ground_truth_answer"].astype(str).tolist()
assert len(GT) == 1002

for name in ["A_top1_single", "B_top1_adaptive_d2.0_score", "C_qg_single", "D_qg_adaptive_d2.0_score"]:
    t = main[main.config == name].sort_values("row").text.fillna(" ").replace("", " ").tolist()
    assert len(t) == 1002
    _, _, F = bscore(t, GT, model_type="roberta-large", lang="en", batch_size=32,
                     device="cuda" if torch.cuda.is_available() else "cpu", verbose=False)
    print(f"{name:32s} BERTScore F1 = {F.mean().item():.4f}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (9 outputs left out)

C_qg_single                      BERTScore F1 = 0.8801


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


D_qg_adaptive_d2.0_score         BERTScore F1 = 0.8640


In [29]:
import pandas as pd
s = pd.read_csv("/kaggle/working/results_span_study/summary_all.csv", index_col=0)
rows = ["A_top1_single","B_top1_adaptive_d2.0_score","C_qg_single","D_qg_adaptive_d2.0_score",
        "E1_ORACLE_best3_sentence_per_candidate","E3_ORACLE_best3_candidate_adaptive_d2.0"]
cols = [c for c in ["F1","BERTScore_F1","ref_recall","cond_recall","words_mean","words_median"] if c in s.columns]
print(s.loc[rows, cols].round(4).to_string())

                                             F1  ref_recall  cond_recall  words_mean  words_median
config                                                                                            
A_top1_single                            0.2961      0.4220       0.4380     55.0090          36.0
B_top1_adaptive_d2.0_score               0.2527      0.4748       0.4964    126.1397          48.5
C_qg_single                              0.2983      0.4241       0.4482     51.5609          36.0
D_qg_adaptive_d2.0_score                 0.2219      0.5292       0.5758    260.6776          70.0
E1_ORACLE_best3_sentence_per_candidate   0.3319      0.4553       0.4928     41.8683          34.0
E3_ORACLE_best3_candidate_adaptive_d2.0  0.2789      0.4768       0.5026     85.0439          42.0
